In [50]:
import os
import pandas as pd
import json
import numpy as np
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

### Database Connection

In [51]:
load_dotenv()

database_url = (
    f"postgresql+psycopg2://{os.environ['DB_USER']}:"
    f"{os.environ['DB_PASSWORD']}@"
    f"{os.environ['DB_HOST']}:"
    f"{os.environ['DB_PORT']}/"
    f"{os.environ['DB_NAME']}"
)

engine = create_engine(
    database_url,
    connect_args={"sslmode": "require"},   
    pool_pre_ping=True,                    
    pool_recycle=3600,                   
)

In [52]:

QUERY_DATABASE = text("""
SELECT title, date, url, text FROM sentiment_data.news WHERE date >= '2025-07-01'  ;
""")
recent_data_news = pd.read_sql(QUERY_DATABASE, engine)


### Helper Functions

In [53]:
import hashlib
import re
import unicodedata


def generate_article_id(title) -> str | pd.NA:
    if pd.isna(title) or not str(title).strip():
        return pd.NA

    normalized_title = unicodedata.normalize("NFKC", str(title))
    normalized_title = normalized_title.casefold().strip()
    normalized_title = re.sub(r"\s+", " ", normalized_title)

    digest = hashlib.sha256(
        normalized_title.encode("utf-8")
    ).hexdigest()[:16]

    return f"article_{digest}"

In [54]:
def assign_trading_date(
    dates: pd.Series,
    cutoff_hour: int = 16,
) -> pd.Series:
    dates = pd.to_datetime(dates, errors="coerce")

    return (
        dates + pd.Timedelta(hours=24 - cutoff_hour)
    ).dt.normalize()

### Time Engineering

In [68]:
data1 = recent_data_news[['title','date']]
data2 = pd.read_csv('articles_rev_eng.csv')

In [69]:
data1['id'] = data1['title'].apply(generate_article_id)

In [70]:
data2['id'] = data2['headline'].apply(generate_article_id)

In [71]:
data1["date"] = pd.to_datetime(data1["date"], errors="coerce")

date_lookup = (
    data1
    .dropna(subset=["id", "date"])
    .drop_duplicates(subset=["id"])
    .set_index("id")["date"]
)

data2["date"] = data2["id"].map(date_lookup)

In [74]:
data2['date'] = assign_trading_date(data2['date'])

In [75]:
data2.to_csv('data2.csv')

In [76]:
data2

,Unnamed: 0,ticker,relevance,sentiment_score,confidence,date,headline,weighted_sentiment,id
0,0,SMGR,1.0,0.50,0.75,2025-07-14,IHSG Awal Pekan Bakal Bertahan di Level 7.000?...,0.37500,article_17ee81da481417c9
1,1,BREN,1.0,0.50,0.75,2025-07-14,IHSG Awal Pekan Bakal Bertahan di Level 7.000?...,0.37500,article_17ee81da481417c9
2,2,PGEO,1.0,0.50,0.75,2025-07-14,IHSG Awal Pekan Bakal Bertahan di Level 7.000?...,0.37500,article_17ee81da481417c9
3,3,INCO,1.0,0.50,0.75,2025-07-14,IHSG Awal Pekan Bakal Bertahan di Level 7.000?...,0.37500,article_17ee81da481417c9
4,4,INDF,1.0,0.50,0.75,2025-07-14,IHSG Awal Pekan Bakal Bertahan di Level 7.000?...,0.37500,article_17ee81da481417c9
...,...,...,...,...,...,...,...,...,...
89364,89364,INDF,0.5,0.25,0.75,2026-07-01,Fitch Ratings Soroti Makroekonomi dan Ketidakp...,0.09375,article_5c19ea9d93c72b70
89365,89365,EXCL,0.5,0.00,0.75,2026-07-01,Fitch Ratings Soroti Makroekonomi dan Ketidakp...,0.00000,article_5c19ea9d93c72b70
89366,89366,BUMI,1.0,0.00,0.75,2026-07-01,Bumi Resources (BUMI) Divestasi Saham Tambang ...,0.00000,article_dbc7076cb27ca60d
89367,89367,BRMS,1.0,0.50,0.75,2026-07-01,Bumi Resources (BUMI) Divestasi Saham Tambang ...,0.37500,article_dbc7076cb27ca60d
